# Data Cleaning & Reporting Automation
### End-to-End Automated Pipeline Demonstration

This interactive notebook demonstrates the complete workflow:
1. **Raw Data Ingestion & Profiling**
2. **Comprehensive Data Validation & Quality Scorecard**
3. **Automated & Configurable Data Cleaning**
4. **Feature Engineering & Transformation**
5. **Dynamic Business KPI Calculation & Statistical Summaries**
6. **Visual Analytics & Chart Generation**
7. **Automated Insights & Recommendations Generation**
8. **Multi-Format Publication-Ready Reporting (Excel, PDF, HTML, CSV)**

In [ ]:
import os
import sys
# Ensure project root is in sys.path
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_data
from src.validator import validate_data
from src.cleaner import clean_data
from src.transformer import transform_data
from src.analyzer import calculate_kpis, generate_statistical_summary, generate_aggregations
from src.insights import generate_insights
from src.report_generator import generate_excel_report, generate_pdf_report, generate_html_report, export_cleaned_data

## 1. Load Raw Messy Dataset
Let's load the sample messy sales dataset containing intentional currency strings, missing values, duplicates, and inconsistent categories.

In [ ]:
raw_df, metadata = load_data('../data/raw/messy_sales_data.csv')
print(f"File Loaded: {metadata['file_name']}")
print(f"Shape: {metadata['rows']} rows, {metadata['columns']} columns")
print(f"Initial Missing Values: {metadata['initial_missing_count']}")
print(f"Initial Duplicate Rows: {metadata['initial_duplicate_count']}")
raw_df.head(10)

## 2. Data Quality Health Assessment
Evaluate overall data quality across Completeness, Uniqueness, Validity, and Consistency dimensions.

In [ ]:
raw_validation = validate_data(raw_df)
print(f"Data Health Score: {raw_validation['quality_score']}/100 [{raw_validation['quality_grade']}]")
print("\nDimensions Breakdown:", raw_validation['dimensions'])
print("\nIdentified Issues:")
for issue in raw_validation['issues_summary']:
    print(f" - {issue}")

## 3. Automated Data Cleaning Pipeline
Execute standardized snake_case naming, exact duplicate removal, currency/type conversion, negative anomaly correction, categorical normalization, and intelligent missing value imputation.

In [ ]:
clean_df, audit_log = clean_data(raw_df)
print(f"Cleaned Shape: {len(clean_df)} rows, {len(clean_df.columns)} columns")
print(f"Duplicates Removed: {audit_log['duplicates_removed']}")
print("\nSteps Executed in Cleaning Pipeline:")
for s in audit_log['steps_executed']:
    print(f" ✔ {s}")
clean_df.head(10)

## 4. Feature Engineering & Transformations
Extract date components (year, month, quarter, day of week) and compute derived business metrics (revenue, discount amount, net revenue).

In [ ]:
transformed_df, transformations = transform_data(clean_df)
print(f"Applied {len(transformations)} transformations:")
for col, desc in transformations.items():
    print(f" + {col}: {desc}")
transformed_df.head(10)

## 5. Post-Cleaning Validation (Before vs After Comparison)

In [ ]:
clean_validation = validate_data(transformed_df)
print(f"Initial Health Score: {raw_validation['quality_score']}/100 [{raw_validation['quality_grade']}]")
print(f"Cleaned Health Score: {clean_validation['quality_score']}/100 [{clean_validation['quality_grade']}]")
print(f"Remaining Missing Values: {clean_validation['missing_report']['total_missing']}")
print(f"Remaining Duplicates: {clean_validation['duplicate_report']['duplicate_count']}")

## 6. Dynamic KPI Intelligence & Statistical Summaries

In [ ]:
kpis = calculate_kpis(transformed_df)
for k, v in kpis.items():
    print(f"{v['label']}: {v['value']} ({v['description']})")

summary_df = generate_statistical_summary(transformed_df)
summary_df

## 7. Dimensional Aggregations & Visualizations

In [ ]:
aggregations = generate_aggregations(transformed_df)
if 'by_category' in aggregations:
    display(aggregations['by_category'])

plt.figure(figsize=(10, 4))
if 'by_category' in aggregations and not aggregations['by_category'].empty:
    cat_df = aggregations['by_category']
    sns.barplot(data=cat_df, x=cat_df.columns[0], y='Total_Value', palette='Blues_r')
    plt.title('Total Revenue by Category', fontsize=12, fontweight='bold')
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()

## 8. Automated Narrative Insights & Recommendations Engine

In [ ]:
insights = generate_insights(raw_validation, clean_validation, audit_log, kpis, aggregations)
print("=== DATA QUALITY INSIGHTS ===")
for item in insights['quality_insights']:
    print(f"✔ {item}")

print("\n=== BUSINESS PERFORMANCE INSIGHTS ===")
for item in insights['business_insights']:
    print(f"💡 {item}")

print("\n=== STRATEGIC RECOMMENDATIONS ===")
for item in insights['recommendations']:
    print(f"🚀 {item}")

## 9. Export Publication-Ready Reports & Clean Dataset

In [ ]:
os.makedirs('../reports', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

# 1. Multi-Sheet Styled Excel Report
excel_path = '../reports/Automated_Sales_Report.xlsx'
generate_excel_report(
    raw_df, transformed_df, raw_validation, clean_validation,
    audit_log, kpis, aggregations, insights, output_path_or_buffer=excel_path
)
print(f"Generated Excel Report: {excel_path} ({os.path.getsize(excel_path):,} bytes)")

# 2. Executive PDF Report
pdf_path = '../reports/Executive_Sales_Report.pdf'
generate_pdf_report(
    transformed_df, raw_validation, clean_validation,
    audit_log, kpis, insights, output_path_or_buffer=pdf_path
)
print(f"Generated PDF Report: {pdf_path} ({os.path.getsize(pdf_path):,} bytes)")

# 3. Standalone HTML Report
html_path = '../reports/Executive_Sales_Report.html'
generate_html_report(
    transformed_df, raw_validation, clean_validation,
    audit_log, kpis, insights, output_path_or_buffer=html_path
)
print(f"Generated HTML Report: {html_path} ({os.path.getsize(html_path):,} bytes)")

# 4. Cleaned CSV Export
csv_clean_path = '../data/processed/cleaned_sales_data.csv'
export_cleaned_data(transformed_df, filepath_or_buffer=csv_clean_path, file_format='csv')
print(f"Exported Cleaned CSV: {csv_clean_path} ({os.path.getsize(csv_clean_path):,} bytes)")